# 03 - 라우팅 기초: if/else와 메서드 분기

이 노트북에서는:
- Kamailio cfg의 제어문을 배웁니다
- `is_method()`로 메시지 종류별로 다르게 처리하는 방법을 배웁니다
- `route[]` 블록의 개념을 이해합니다

---

## Kamailio cfg의 기본 구조

Kamailio cfg는 세 가지 섹션으로 구성됩니다:

```kamailio
# 1. Global Parameters
listen=udp:10.0.0.1:5060

# 2. Modules
loadmodule "sl.so"
loadmodule "tm.so"

# 3. Routing Logic
request_route {
    if (is_method("REGISTER")) {
        save("location");
    }
    if (is_method("INVITE")) {
        record_route();
        t_relay();
    }
}
```

이 노트북에서는 **3번 라우팅 로직** 부분을 연습합니다.

## 1. if/else 기본

Kamailio의 if/else는 C언어와 비슷합니다.

In [1]:
%%sip INVITE
From: <sip:1001@example.com>;tag=test1
To: <sip:1002@example.com>

Mock INVITE message created:


INVITE sip:1002@example.com SIP/2.0
From: <sip:1001@example.com>;tag=test1
To: <sip:1002@example.com>
Call-ID: a2bf94d3@mock
CSeq: 1 INVITE


Variables initialized:


  $Ri = "127.0.0.1"


  $Rp = 5060


...

  $ru = "sip:1002@example.com"


  $si = "127.0.0.1"


  $sp = 5060


  $tu = "sip:1002@example.com"


  $ua = ""


In [2]:
# 메서드 확인
if (is_method("INVITE")) {
    xlog("This is an INVITE request");
}

✓ if (is_method("INVITE")) → TRUE


[LOG] This is an INVITE request


  🔀 if(is_method("INVITE")): TRUE


In [3]:
# 이건 FALSE가 되어야 합니다
if (is_method("REGISTER")) {
    xlog("This won't be printed");
}

✗ if (is_method("REGISTER")) → FALSE


  🔀 if(is_method("REGISTER")): FALSE


## 2. if/else 분기

메서드별로 다른 처리를 할 수 있습니다.

In [4]:
# 변수 비교
$var(method) = "INVITE";

if ($var(method) == "INVITE") {
    xlog("Processing INVITE");
} else {
    xlog("Not INVITE");
}

$var(method) = "INVITE"  (string)


✓ if ($var(method) == "INVITE") → TRUE


[LOG] Processing INVITE


  🔀 if($var(method) == "INVITE"): TRUE


Variables changed:


  $var(method): <null> → "INVITE"


## 3. has_totag() — 다이얼로그 내 요청 구분

`has_totag()`는 To 헤더에 tag가 있는지 확인합니다.
- **초기 요청** (INVITE, REGISTER): tag가 없음 → FALSE
- **다이얼로그 내 요청** (BYE, re-INVITE): tag가 있음 → TRUE

이게 왜 중요하냐면, 이미 통화 중인 세션의 요청은 다르게 처리해야 하기 때문입니다.

In [5]:
# 초기 INVITE: To tag 없음
%%sip INVITE
From: <sip:1001@example.com>;tag=from1
To: <sip:1002@example.com>

Mock INVITE message created:


INVITE sip:1002@example.com SIP/2.0
From: <sip:1001@example.com>;tag=from1
To: <sip:1002@example.com>
Call-ID: d7e6cae4@mock
CSeq: 1 INVITE


Variables initialized:


  $Ri = "127.0.0.1"


  $Rp = 5060


...

  $ru = "sip:1002@example.com"


  $si = "127.0.0.1"


  $sp = 5060


  $tu = "sip:1002@example.com"


  $ua = ""


In [6]:
# 초기 INVITE이므로 has_totag() → FALSE
if (has_totag()) {
    xlog("In-dialog request");
} else {
    xlog("Initial request — needs routing");
}

✗ if (has_totag()) → FALSE → else


[LOG] Initial request — needs routing


  🔀 if(has_totag()): FALSE → else


## 4. 부정 연산자 (!)

`!` 연산자로 조건을 부정할 수 있습니다.

In [7]:
# REGISTER가 아니면 (즉, INVITE이므로 TRUE)
if (!is_method("REGISTER")) {
    xlog("Not a REGISTER — proceed with routing");
}

✓ if (!is_method("REGISTER")) → TRUE


[LOG] Not a REGISTER — proceed with routing


  🔀 if(!is_method("REGISTER")): TRUE


## 5. route() 호출

Kamailio에서 `route(name)`은 다른 라우팅 블록을 호출합니다.
코드를 모듈화하는 방법입니다.

In [8]:
# 실제 Kamailio에서는 route[REGISTRAR] 블록이 따로 정의되어 있겠지만,
# 여기서는 route() 호출만 시뮬레이션합니다.
if (is_method("REGISTER")) {
    route(REGISTRAR);
} else {
    route(LOCATION);
}

✗ if (is_method("REGISTER")) → FALSE → else


→ route(LOCATION) called


  🔀 if(is_method("REGISTER")): FALSE → else


## 6. drop / exit

- `drop`: 메시지 처리를 즉시 중단하고 버림
- `exit`: 메시지 처리를 중단 (drop과 유사)
- `return`: 현재 route 블록에서 빠져나감

In [9]:
$var(blocked) = 1;

if ($var(blocked) == 1) {
    xlog("Blocked user — dropping message");
    drop;
} else {
    xlog("Allowed");
}

$var(blocked) = 1  (integer)


✓ if ($var(blocked) == 1) → TRUE


[LOG] Blocked user — dropping message


→ drop executed


  🔀 if($var(blocked) == 1): TRUE


Variables changed:


  $var(blocked): <null> → 1


## 7. send_reply — 응답 보내기

SIP 응답 코드를 직접 보낼 수 있습니다.

| 코드 | 의미 |
|------|------|
| 200 | OK |
| 401 | Unauthorized |
| 403 | Forbidden |
| 404 | Not Found |
| 503 | Service Unavailable |

In [10]:
$var(user_exists) = 0;

if ($var(user_exists) == 0) {
    send_reply(404, "Not Found");
} else {
    send_reply(200, "OK");
}

$var(user_exists) = 0  (integer)


✓ if ($var(user_exists) == 0) → TRUE


→ send_reply(404, "Not Found")


  🔀 if($var(user_exists) == 0): TRUE


Variables changed:


  $var(user_exists): <null> → 0


---

### 요약

| 개념 | 문법 |
|------|------|
| 조건문 | `if (condition) { ... } else { ... }` |
| 메서드 확인 | `is_method("INVITE")` |
| To tag 확인 | `has_totag()` |
| 부정 | `!condition` |
| 비교 | `==`, `!=`, `>`, `<` |
| 라우팅 블록 호출 | `route(NAME);` |
| 메시지 버리기 | `drop;` or `exit;` |
| 응답 전송 | `send_reply(code, "reason");` |

다음 노트북: **Intermediate/01-transformations.ipynb** →